# Generateur de README Professionnel
### Outil DA/DS - Documenter un projet pour GitHub
> Ce notebook genere automatiquement un README.md professionnel
> base sur tes resultats EDA Heart Disease

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datetime import date

sns.set_theme(style='whitegrid', font_scale=1.1)
np.random.seed(42)

# Charger le dataset
df = pd.read_csv('/home/joachin/formation_data/heart.csv')
print(f'Dataset : {df.shape[0]} patients x {df.shape[1]} variables')

Dataset : 1025 patients x 14 variables


---
## ETAPE 1 - Calcul des statistiques pour le README

In [2]:
# Toutes les stats dont le README a besoin
n_patients      = len(df)
n_variables     = df.shape[1]
n_manquants     = df.isnull().sum().sum()
pct_malade      = df['target'].mean() * 100
n_malade        = df['target'].sum()
n_sain          = len(df) - n_malade
age_moyen       = df['age'].mean()
pct_homme       = df['sex'].mean() * 100

malade          = df[df['target']==1]
non_malade      = df[df['target']==0]
fc_malade       = malade['thalach'].mean()
fc_sain         = non_malade['thalach'].mean()

# Correlations avec target
corr_target = df.corr()['target'].drop('target').sort_values(ascending=False)
risques     = corr_target[corr_target > 0]
protecteurs = corr_target[corr_target < 0]

# Tests statistiques
tests_t = {}
vars_cont = ['age','trestbps','chol','thalach','oldpeak']
for var in vars_cont:
    t, p = stats.ttest_ind(malade[var], non_malade[var])
    tests_t[var] = p

tests_chi2 = {}
vars_cat = ['sex','cp','fbs','restecg','exang','slope','ca','thal']
for var in vars_cat:
    table = pd.crosstab(df[var], df['target'])
    chi2, p, _, _ = stats.chi2_contingency(table)
    tests_chi2[var] = p

def sig(p):
    return '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'

print('Stats calculees - pret pour le README')

Stats calculees - pret pour le README


---
## ETAPE 2 - Generation du README.md

In [3]:
# Generer le contenu du README dynamiquement
top_risque = risques.head(4)
top_protec = protecteurs.head(6)

readme = f'''# Heart Disease Risk Analysis — UCI Dataset

![Python](https://img.shields.io/badge/Python-3.11-blue)
![Status](https://img.shields.io/badge/Status-Complete-green)
![Dataset](https://img.shields.io/badge/Dataset-Kaggle_UCI-orange)

## Overview

Complete exploratory data analysis (EDA) on the UCI Heart Disease dataset.
The goal is to identify the main risk factors associated with heart disease
using statistical methods and machine learning pipelines.

---

## Dataset

| Property | Value |
|---|---|
| Source | Kaggle — Heart Disease UCI |
| Patients | {n_patients} |
| Variables | {n_variables} |
| Missing values | {n_manquants} |
| Target | `target` (1=Heart disease, 0=Healthy) |
| Class balance | {pct_malade:.1f}% positive / {100-pct_malade:.1f}% negative |
| Male patients | {pct_homme:.0f}% |
| Mean age | {age_moyen:.0f} years |

---

## Methodology — 8 Steps

| Step | Description | Tools |
|---|---|---|
| 1 | Data overview : shape, dtypes, missing values | pandas describe() |
| 2 | Continuous variables : histogram + KDE + box plot | matplotlib seaborn |
| 3 | Categorical variables : frequency charts | pandas value_counts |
| 4 | Bivariate analysis : distributions by target class | scipy t-test chi2 |
| 5 | Correlation matrix Pearson | seaborn heatmap |
| 6 | Statistical tests on all variables | scipy.stats |
| 7 | Pairplot top 4 variables | seaborn pairplot |
| 8 | Final dashboard 6 panels | matplotlib gridspec |

---

## Key Findings

### Risk Factors (positive correlation with disease)

| Variable | Pearson r | Interpretation |
|---|---|---|
'''

interp_risque = {
    'cp'     : 'Chest pain type strongly linked to disease',
    'thalach': 'Higher max heart rate linked to disease',
    'slope'  : 'ST segment slope pattern',
    'restecg': 'Abnormal ECG results'
}
for var, val in top_risque.items():
    interp = interp_risque.get(var, 'Positive association with disease')
    readme += f'| `{var}` | +{val:.3f} | {interp} |\n'

readme += '''
### Protective Factors (negative correlation with disease)

| Variable | Pearson r | Interpretation |
|---|---|---|
'''

interp_protec = {
    'oldpeak': 'Lower ST depression = healthier profile',
    'exang'  : 'No exercise-induced angina = healthier',
    'ca'     : 'Fewer colored vessels = healthier',
    'thal'   : 'Normal thalassemia = healthier',
    'sex'    : 'Female patients less affected',
    'age'    : 'Younger patients less affected'
}
for var, val in top_protec.items():
    interp = interp_protec.get(var, 'Negative association with disease')
    readme += f'| `{var}` | {val:.3f} | {interp} |\n'

readme += f'''
### Key Observations
- Diseased patients have significantly higher max heart rate : **{fc_malade:.0f} vs {fc_sain:.0f} bpm** (p < 0.001)
- Chest pain type 2 is the strongest categorical predictor
- Male patients represent **{pct_homme:.0f}%** of the dataset
- Dataset is well-balanced : **{pct_malade:.1f}%** positive cases
- No missing values — dataset ready for modeling

---

## Statistical Tests Summary

| Variable | Test | p-value | Significance |
|---|---|---|---|
'''

for var, p in tests_t.items():
    readme += f'| `{var}` | t-test | {p:.4f} | {sig(p)} |\n'
for var, p in tests_chi2.items():
    readme += f'| `{var}` | chi-squared | {p:.4f} | {sig(p)} |\n'

readme += '''
---

## Visualizations

### EDA Dashboard
![Dashboard EDA](dashboard_heart_eda.png)

---

## Project Structure

```
projet_diabete_madagascar/
    README.md                        <- This file
    notebook_eda_heart_08.ipynb      <- Complete EDA notebook
    notebook_readme_09.ipynb         <- This README generator
    notebook_projet_07.ipynb         <- Full ML pipeline on PostgreSQL data
    heart.csv                        <- UCI Heart Disease dataset
    dashboard_heart_eda.png          <- EDA dashboard
    dashboard_diabete.png            <- Diabetes analysis dashboard
    predictions_diabete.csv          <- ML model predictions
```

---

## Technologies

```
Python 3.11 | pandas | numpy | matplotlib | seaborn
scipy | statsmodels | scikit-learn
PostgreSQL 18 | psycopg2 | SQLAlchemy
```

---

## How to Run

```bash
git clone https://github.com/odijoa5-create/projet_diabete_madagascar.git
cd projet_diabete_madagascar
conda create -n formation_data python=3.11
conda activate formation_data
pip install pandas numpy matplotlib seaborn scipy statsmodels scikit-learn jupyter
jupyter notebook
```

---

## Next Steps

- [ ] Machine Learning modeling (Random Forest, Gradient Boosting)
- [ ] Hyperparameter tuning with GridSearchCV
- [ ] SHAP values for model explainability
- [ ] SIR/SEIR epidemiological model
- [ ] Streamlit interactive dashboard

---

## Author

**Joachin** — Data Analyst / Data Scientist
Master in Numerical Analysis | PhD candidate in Computational Epidemiology
GitHub : [odijoa5-create](https://github.com/odijoa5-create)
'''

# Sauvegarder le README
with open('/home/joachin/formation_data/projet_diabete_madagascar/README.md', 'w') as f:
    f.write(readme)

print('README.md genere et sauvegarde !')
print(f'Taille : {len(readme)} caracteres')
print(f'Lignes : {readme.count(chr(10))}')

README.md genere et sauvegarde !
Taille : 4640 caracteres
Lignes : 156


---
## ETAPE 3 - Verifier et pousser sur GitHub

In [4]:
# Afficher un apercu du README
with open('/home/joachin/formation_data/projet_diabete_madagascar/README.md', 'r') as f:
    contenu = f.read()

print('APERCU DES 50 PREMIERES LIGNES :')
print('-'*60)
for ligne in contenu.split('\n')[:50]:
    print(ligne)

APERCU DES 50 PREMIERES LIGNES :
------------------------------------------------------------
# Heart Disease Risk Analysis — UCI Dataset

![Python](https://img.shields.io/badge/Python-3.11-blue)
![Status](https://img.shields.io/badge/Status-Complete-green)
![Dataset](https://img.shields.io/badge/Dataset-Kaggle_UCI-orange)

## Overview

Complete exploratory data analysis (EDA) on the UCI Heart Disease dataset.
The goal is to identify the main risk factors associated with heart disease
using statistical methods and machine learning pipelines.

---

## Dataset

| Property | Value |
|---|---|
| Source | Kaggle — Heart Disease UCI |
| Patients | 1025 |
| Variables | 14 |
| Missing values | 0 |
| Target | `target` (1=Heart disease, 0=Healthy) |
| Class balance | 51.3% positive / 48.7% negative |
| Male patients | 70% |
| Mean age | 54 years |

---

## Methodology — 8 Steps

| Step | Description | Tools |
|---|---|---|
| 1 | Data overview : shape, dtypes, missing values | pandas describe() |

In [5]:
# Commandes git a executer dans le terminal
print('COMMANDES A EXECUTER DANS LE TERMINAL :')
print('='*55)
print()
print('cd ~/formation_data/projet_diabete_madagascar')
print('cp ~/formation_data/notebook_eda_heart_08.ipynb .')
print('cp ~/formation_data/heart.csv .')
print('cp ~/formation_data/dashboard_heart_eda.png .')
print('cp ~/formation_data/dashboard_diabete.png .')
print('cp ~/formation_data/predictions_diabete.csv .')
print('git add .')
print('git commit -m "EDA Heart Disease + README professionnel auto-genere"')
print('git push')
print()
print('Ton portfolio sera visible sur :')
print('https://github.com/odijoa5-create/projet_diabete_madagascar')

COMMANDES A EXECUTER DANS LE TERMINAL :

cd ~/formation_data/projet_diabete_madagascar
cp ~/formation_data/notebook_eda_heart_08.ipynb .
cp ~/formation_data/heart.csv .
cp ~/formation_data/dashboard_heart_eda.png .
cp ~/formation_data/dashboard_diabete.png .
cp ~/formation_data/predictions_diabete.csv .
git add .
git commit -m "EDA Heart Disease + README professionnel auto-genere"
git push

Ton portfolio sera visible sur :
https://github.com/odijoa5-create/projet_diabete_madagascar
